# Glove Gesture ML Explorer

Configurable pipeline for exploring hand gesture recognition from the sensory glove dataset.

**Workflow:**
1. [Config](#1-configuration) — Set data path, select columns, labels, and splits
2. [Load Data](#2-data-loading) — Read CSVs, attach labels from folder names
3. [Preprocess](#3-preprocessing) — Filter, normalise, extract features
4. [Train & Evaluate](#4-training--evaluation) — Run selected ML algorithms
5. [Results](#5-results-comparison) — Compare algorithms side-by-side

---
**Dataset structure:**  
Each folder under `DATA_ROOT` is a gesture label (e.g. `TwoHand_L_Fist_R_Fist`).  
Each CSV = one 5-second trial at ~22 Hz → ~110 rows × 295 columns.

> **Note on `TwoHand_L_Flat_R_Flat`:** Files in this folder are named `*L_Fist_R_Fist*` by mistake — they are Flat gestures. The label is taken from the **folder name**, not the filename, so this is handled correctly automatically.

---
## 0. Install / Import Dependencies

In [ ]:
# Run once to install any missing packages
import subprocess, sys
pkgs = ['scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencies ready.')

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

# Classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
print('Imports OK.')

---
## 1. Configuration

**Edit this cell to control the entire pipeline.**

In [ ]:
# =============================================================================
# 1A. DATA PATH
# =============================================================================
# Path to the root folder containing gesture subfolders (ThesisA.data).
# Each subfolder name becomes the class label.
# Accepts absolute paths or paths relative to this notebook.
DATA_ROOT = '../ML/TwoHand_L_Fist_R_Fist_filtered_butterworth_lp'   # <-- change this to point to your ThesisA.data folder

# Labels to INCLUDE. Set to None to use ALL subfolders automatically.
# Example: INCLUDE_LABELS = ['TwoHand_L_Fist_R_Fist', 'TwoHand_L_Flat_R_Flat']
INCLUDE_LABELS = None

# =============================================================================
# 1B. COLUMN SELECTION
# =============================================================================
# Choose which sensor modalities to include.
# Each flag independently includes/excludes that group of columns.

USE_YAW_PITCH_ROLL   = True    # IMU Euler angles (yaw, pitch, roll / heading)
USE_QUATERNIONS      = False   # Raw quaternion components (quat_w/x/y/z)
                               # Note: for TwoHand_L_Flat_R_Flat, quaternions are
                               #       the primary data — pitch/roll/yaw are derived.
                               #       Set True if using that label.
USE_ACCELEROMETER    = True    # Linear accelerometer (ax, ay, az)
USE_FLEX_SENSORS     = True    # Flex sensor readings (mcp_flex, pip_flex)

# Choose which body segments to include.
# Available: 'palm_prox', 'thumb', 'index', 'middle', 'ring', 'pinky', 'wrist'
# 'palm_mid'  = excluded — IMU columns exist in CSV but contain no sensor data
# 'palm_prox' = IMU on back of palm (proximal, near wrist)
# 'wrist'     = IMU at end of forearm closest to wrist
# Note: flex sensors exist for fingers only (thumb, index, middle, ring, pinky) — NOT palm
INCLUDE_SEGMENTS = [
    # 'palm_mid',  # excluded: IMU columns exist but contain no data
    'palm_prox',
    'thumb',
    'index',
    'middle',
    'ring',
    'pinky',
    'wrist',
]

# Choose which hands to include
INCLUDE_HANDS = ['left', 'right']  # options: 'left', 'right', or both

# =============================================================================
# 1C. PREPROCESSING
# =============================================================================

# --- Resampling ---
# Resample each trial to a fixed number of time steps (handles variable-length files).
# Set to None to skip resampling (all files must then have the same row count).
RESAMPLE_TO_N_STEPS = 100       # target number of rows per trial

# --- Low-pass Butterworth filter ---
APPLY_BUTTERWORTH    = True     # Apply low-pass filter to smooth IMU noise
BUTTERWORTH_CUTOFF   = 5.0      # Cutoff frequency in Hz
BUTTERWORTH_ORDER    = 4        # Filter order
SAMPLING_RATE_HZ     = 22.0     # Approximate sampling rate of the glove

# --- Normalisation ---
# 'standard'  → zero mean, unit variance (StandardScaler) — good for SVM, LDA
# 'minmax'    → scale to [0, 1]  (MinMaxScaler) — good for KNN, neural nets
# None        → no normalisation
NORMALISATION = 'standard'

# --- Feature extraction mode ---
# 'flatten'    → use the raw resampled time series as a flat feature vector
#                (n_steps × n_channels features per sample)
# 'stats'      → extract statistical features per channel
#                (mean, std, min, max, range, RMS, skew, kurtosis)
# 'fft'        → FFT magnitude spectrum per channel
# 'stats+fft'  → combine statistical + FFT features
FEATURE_MODE = 'stats'    # Recommended starting point

# =============================================================================
# 1D. TRAIN / TEST SPLIT
# =============================================================================
TEST_SIZE        = 0.2    # Fraction of data held out for testing
RANDOM_STATE     = 42
CV_FOLDS         = 5      # Number of cross-validation folds

# =============================================================================
# 1E. ALGORITHMS
# =============================================================================
# Set True/False to include/exclude each algorithm.
ALGORITHMS = {
    'SVM (RBF)'           : True,
    'SVM (Linear)'        : True,
    'Random Forest'       : True,
    'KNN'                 : True,
    'Gradient Boosting'   : False,   # Slower — enable when needed
    'Logistic Regression' : True,
    'LDA'                 : True,
}

print('Configuration loaded.')
print(f'  Data root  : {DATA_ROOT}')
print(f'  Hands      : {INCLUDE_HANDS}')
print(f'  Segments   : {INCLUDE_SEGMENTS}')
print(f'  Modalities : YPR={USE_YAW_PITCH_ROLL}, Quat={USE_QUATERNIONS}, Accel={USE_ACCELEROMETER}, Flex={USE_FLEX_SENSORS}')
print(f'  Features   : {FEATURE_MODE}')
print(f'  Algorithms : {[k for k,v in ALGORITHMS.items() if v]}')

---
## 2. Data Loading

In [ ]:
# ── Build column selector from config ─────────────────────────────────────────

# All 295 columns, grouped by type for easy selection
META_COLS = [
    'run_index', 'request_id', 'request_ts',
    'right_recv_time_ms', 'right_glove_time_ms', 'right_time',
    'left_recv_time_ms',  'left_glove_time_ms',  'left_time',
    'left_hand', 'right_hand'
]

SEGMENTS_WITH_FLEX = ['thumb', 'index', 'middle', 'ring', 'pinky']  # palm has no flex sensors
IMU_LOCATIONS = ['mid', 'prox']  # distal phalanx / proximal phalanx

def build_sensor_columns(hands, segments, use_ypr, use_quat, use_accel, use_flex):
    """Return list of sensor column names matching the user config."""
    cols = []
    for hand in hands:
        for seg in segments:
            # Determine IMU sub-locations for this segment
            # palm has mid + prox; all finger segments have mid + prox; wrist has no sub-location
            if seg == 'wrist':
                prefix = f'{hand}_wrist'
                if use_ypr:
                    cols += [f'{prefix}_heading', f'{prefix}_pitch', f'{prefix}_roll']
                if use_quat:
                    cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                             f'{prefix}_quat_y', f'{prefix}_quat_z']
                if use_accel:
                    cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # wrist has no flex sensor
            else:
                # segment has mid + prox IMU positions
                # For 'thumb','index' etc, map to e.g. 'left_thumb_mid_yaw'
                for loc in ['mid', 'prox']:
                    prefix = f'{hand}_{seg}_{loc}'
                    if use_ypr:
                        cols += [f'{prefix}_yaw', f'{prefix}_pitch', f'{prefix}_roll']
                    if use_quat:
                        cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                                 f'{prefix}_quat_y', f'{prefix}_quat_z']
                    if use_accel:
                        cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # Flex sensors (one per finger joint combination, not per loc)
                if use_flex and seg in ['thumb', 'index', 'middle', 'ring', 'pinky']:  # palm has no flex
                    cols += [f'{hand}_{seg}_mcp_flex', f'{hand}_{seg}_pip_flex']
    return cols

# Map user-friendly segment names to CSV naming convention
seg_map = {
    'palm_mid':  'palm',   # excluded by default — no sensor data on palm_mid
    'palm_prox': 'palm',   # deduplicated below
    'thumb':     'thumb',
    'index':     'index',
    'middle':    'middle',
    'ring':      'ring',
    'pinky':     'pinky',
    'wrist':     'wrist',
}

# Resolve segments, de-duplicating palm if both palm_mid and palm_prox selected
resolved_segs = list(dict.fromkeys([seg_map[s] for s in INCLUDE_SEGMENTS]))

SENSOR_COLS = build_sensor_columns(
    hands    = INCLUDE_HANDS,
    segments = resolved_segs,
    use_ypr  = USE_YAW_PITCH_ROLL,
    use_quat = USE_QUATERNIONS,
    use_accel= USE_ACCELEROMETER,
    use_flex = USE_FLEX_SENSORS
)

print(f'Selected {len(SENSOR_COLS)} sensor columns.')
print('First 10:', SENSOR_COLS[:10])
print('Last 10:', SENSOR_COLS[-10:])

In [ ]:
# ── Load all CSV files ─────────────────────────────────────────────────────────

def load_dataset(data_root, include_labels, sensor_cols):
    """Load all CSVs from label subfolders. Returns list of (df_trial, label)."""
    data_root = os.path.expanduser(data_root)
    if not os.path.isdir(data_root):
        raise FileNotFoundError(
            f"DATA_ROOT not found: '{data_root}'\n"
            "Please update DATA_ROOT in the Configuration cell to point to your "
            "ThesisA.data folder."
        )

    label_dirs = sorted([
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
        and d not in ('PDF', 'sdb', 'idk')  # skip non-gesture folders
    ])

    if include_labels is not None:
        label_dirs = [d for d in label_dirs if d in include_labels]

    if not label_dirs:
        raise ValueError('No label folders found. Check DATA_ROOT and INCLUDE_LABELS.')

    print(f'Found {len(label_dirs)} gesture classes:')
    trials, labels = [], []

    for label in label_dirs:
        folder = os.path.join(data_root, label)
        csv_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
        print(f'  [{label}]  →  {len(csv_files)} files')
        for fpath in csv_files:
            try:
                df = pd.read_csv(fpath)
                # Keep only sensor columns that exist in this file
                available = [c for c in sensor_cols if c in df.columns]
                if not available:
                    print(f'    WARNING: no matching sensor columns in {os.path.basename(fpath)}')
                    continue
                trials.append(df[available].values.astype(np.float32))
                labels.append(label)
            except Exception as e:
                print(f'    ERROR loading {os.path.basename(fpath)}: {e}')

    print(f'\nTotal trials loaded: {len(trials)}')
    return trials, labels, label_dirs

trials_raw, labels_raw, class_names = load_dataset(
    DATA_ROOT, INCLUDE_LABELS, SENSOR_COLS
)

# Class distribution
from collections import Counter
print('\nClass distribution:')
for cls, cnt in sorted(Counter(labels_raw).items()):
    print(f'  {cls}: {cnt} trials')

---
## 3. Preprocessing

In [ ]:
# ── Step 1: Resample to fixed length ──────────────────────────────────────────

def resample_trial(trial, n_steps):
    """Resample a (T, C) array to (n_steps, C) using linear interpolation."""
    T, C = trial.shape
    if T == n_steps:
        return trial
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    resampled = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        resampled[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return resampled

if RESAMPLE_TO_N_STEPS is not None:
    trials_resampled = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials_raw]
    print(f'Resampled all trials to {RESAMPLE_TO_N_STEPS} time steps.')
else:
    trials_resampled = trials_raw
    lengths = [t.shape[0] for t in trials_resampled]
    print(f'No resampling. Trial lengths: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}')
    if min(lengths) != max(lengths):
        print('  WARNING: Variable-length trials detected. Enable RESAMPLE_TO_N_STEPS '
              'or use a feature extraction mode that handles variable length.')

print(f'Trial shape: {trials_resampled[0].shape}  (time_steps × channels)')

In [ ]:
# ── Step 2: Butterworth low-pass filter ───────────────────────────────────────

def apply_butterworth(trials, cutoff, order, fs):
    """Apply a zero-phase Butterworth low-pass filter to each channel of each trial."""
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    filtered = []
    for t in trials:
        t_filt = scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32)
        filtered.append(t_filt)
    return filtered

if APPLY_BUTTERWORTH:
    trials_filtered = apply_butterworth(
        trials_resampled, BUTTERWORTH_CUTOFF, BUTTERWORTH_ORDER, SAMPLING_RATE_HZ
    )
    print(f'Applied Butterworth low-pass filter: {BUTTERWORTH_CUTOFF} Hz cutoff, '
          f'order {BUTTERWORTH_ORDER}, Nyquist = {SAMPLING_RATE_HZ/2} Hz')
else:
    trials_filtered = trials_resampled
    print('Butterworth filter skipped.')

In [ ]:
# ── Step 3: Feature extraction ────────────────────────────────────────────────

def extract_stats(trial):
    """Statistical features per channel: mean, std, min, max, range, RMS, skew, kurtosis."""
    feats = np.concatenate([
        trial.mean(axis=0),
        trial.std(axis=0),
        trial.min(axis=0),
        trial.max(axis=0),
        trial.max(axis=0) - trial.min(axis=0),          # range
        np.sqrt((trial**2).mean(axis=0)),                # RMS
        skew(trial, axis=0).astype(np.float32),
        kurtosis(trial, axis=0).astype(np.float32),
    ])
    return feats

def extract_fft(trial, fs):
    """FFT magnitude features per channel (first half of spectrum)."""
    n = trial.shape[0]
    fft_mag = np.abs(np.fft.rfft(trial, axis=0))  # shape: (n//2+1, C)
    return fft_mag.flatten().astype(np.float32)

def extract_features(trials, mode, fs):
    X = []
    for t in trials:
        if mode == 'flatten':
            feats = t.flatten()
        elif mode == 'stats':
            feats = extract_stats(t)
        elif mode == 'fft':
            feats = extract_fft(t, fs)
        elif mode == 'stats+fft':
            feats = np.concatenate([extract_stats(t), extract_fft(t, fs)])
        else:
            raise ValueError(f'Unknown FEATURE_MODE: {mode}')
        X.append(feats)
    return np.array(X, dtype=np.float32)

X = extract_features(trials_filtered, FEATURE_MODE, SAMPLING_RATE_HZ)
print(f'Feature matrix shape: {X.shape}  ({X.shape[0]} trials × {X.shape[1]} features)')

# Encode labels
le = LabelEncoder()
y = le.fit_transform(labels_raw)
print(f'Labels encoded: {dict(zip(le.classes_, range(len(le.classes_))))}')

In [ ]:
# ── Step 4: Normalise + split ─────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

if NORMALISATION == 'standard':
    scaler = StandardScaler()
elif NORMALISATION == 'minmax':
    scaler = MinMaxScaler()
else:
    scaler = None

if scaler is not None:
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    print(f'Applied {NORMALISATION} normalisation.')
else:
    print('No normalisation applied.')

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

In [ ]:
# ── Quick data visualisation: class distribution ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
label_counts = pd.Series(labels_raw).value_counts()
axes[0].barh(label_counts.index, label_counts.values, color='steelblue')
axes[0].set_xlabel('Number of trials')
axes[0].set_title('Class Distribution')
for i, v in enumerate(label_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center')

# Sample trial: first two channels
sample = trials_filtered[0]
t_axis = np.linspace(0, 5, sample.shape[0])
axes[1].plot(t_axis, sample[:, 0], label=SENSOR_COLS[0], alpha=0.8)
if sample.shape[1] > 1:
    axes[1].plot(t_axis, sample[:, 1], label=SENSOR_COLS[1], alpha=0.8)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Value')
axes[1].set_title(f'Sample Trial — {labels_raw[0]}')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 4. Training & Evaluation

In [ ]:
# ── Define classifiers from config ────────────────────────────────────────────

CLASSIFIER_DEFS = {
    'SVM (RBF)':           SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE),
    'SVM (Linear)':        SVC(kernel='linear', C=1, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LDA':                 LinearDiscriminantAnalysis(),
}

active_classifiers = {k: v for k, v in CLASSIFIER_DEFS.items() if ALGORITHMS.get(k, False)}
print(f'Running {len(active_classifiers)} algorithms: {list(active_classifiers.keys())}')

In [ ]:
# ── Train, cross-validate, and test each classifier ───────────────────────────

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, clf in active_classifiers.items():
    print(f'\n--- {name} ---')

    # Cross-validation on training set
    cv_scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f'  CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

    # Fit on full training set
    clf.fit(X_train, y_train)

    # Test set evaluation
    y_pred = clf.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    print(f'  Test accuracy: {test_acc:.4f}')

    results[name] = {
        'cv_mean':    cv_scores.mean(),
        'cv_std':     cv_scores.std(),
        'test_acc':   test_acc,
        'y_pred':     y_pred,
        'clf':        clf,
    }

print('\n✓ All classifiers trained.')

In [ ]:
# ── Per-class classification reports ─────────────────────────────────────────

for name, res in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=le.classes_,
        zero_division=0
    ))

---
## 5. Results Comparison

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

summary = pd.DataFrame({
    'Algorithm':    list(results.keys()),
    'CV Acc (mean)': [f"{r['cv_mean']:.4f}" for r in results.values()],
    'CV Acc (±std)': [f"{r['cv_std']:.4f}"  for r in results.values()],
    'Test Acc':      [f"{r['test_acc']:.4f}" for r in results.values()],
})
summary = summary.sort_values('Test Acc', ascending=False).reset_index(drop=True)
print('\n=== Results Summary ===')
print(summary.to_string(index=False))

In [ ]:
# ── Bar chart: CV vs Test accuracy ────────────────────────────────────────────

names      = list(results.keys())
cv_means   = [results[n]['cv_mean']  for n in names]
cv_stds    = [results[n]['cv_std']   for n in names]
test_accs  = [results[n]['test_acc'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(names) * 1.5), 5))
bars1 = ax.bar(x - width/2, cv_means, width, yerr=cv_stds,
               label='CV Accuracy', color='steelblue', capsize=4, alpha=0.85)
bars2 = ax.bar(x + width/2, test_accs, width,
               label='Test Accuracy', color='coral', alpha=0.85)

ax.set_xlabel('Algorithm')
ax.set_ylabel('Accuracy')
ax.set_title(f'Algorithm Comparison\n'
             f'Features: {FEATURE_MODE} | '
             f'Preprocessing: Butterworth={APPLY_BUTTERWORTH}, Norm={NORMALISATION}')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices for all active classifiers ────────────────────────────

n_clf = len(results)
ncols = min(3, n_clf)
nrows = (n_clf + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten() if n_clf > 1 else [axes]

short_labels = [lbl.replace('TwoHand_', '').replace('TwoHandDynamic_', 'Dyn_') for lbl in le.classes_]

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=short_labels, yticklabels=short_labels,
                linewidths=0.5)
    ax.set_title(f'{name}\n(Test Acc: {res["test_acc"]:.3f})', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

# Hide unused subplots
for ax in axes[n_clf:]:
    ax.set_visible(False)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance (Random Forest) ───────────────────────────────────────

if 'Random Forest' in results:
    rf = results['Random Forest']['clf']
    importances = rf.feature_importances_

    # Map feature indices back to channel names
    n_ch = len(SENSOR_COLS)
    stat_names = ['mean', 'std', 'min', 'max', 'range', 'rms', 'skew', 'kurtosis']

    if FEATURE_MODE == 'stats':
        # importances shape: (8 * n_ch,) — group by channel
        ch_importance = importances.reshape(len(stat_names), n_ch).sum(axis=0)
        top_n = 20
        top_idx = np.argsort(ch_importance)[::-1][:top_n]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh([SENSOR_COLS[i] for i in top_idx[::-1]],
                ch_importance[top_idx[::-1]], color='forestgreen', alpha=0.8)
        ax.set_xlabel('Summed Importance (across stat features)')
        ax.set_title(f'Top {top_n} Most Important Channels — Random Forest')
        plt.tight_layout()
        plt.show()
    else:
        top_n = 30
        top_idx = np.argsort(importances)[::-1][:top_n]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(range(top_n), importances[top_idx], color='forestgreen', alpha=0.8)
        ax.set_xlabel('Feature index (top 30)')
        ax.set_ylabel('Importance')
        ax.set_title('Top 30 Feature Importances — Random Forest')
        plt.tight_layout()
        plt.show()
else:
    print('Random Forest not enabled — skipping feature importance plot.')

---
## 6. Experiment Log

Use this cell to record what you tried and what worked.

| # | Hands | Segments | Modalities | Feature Mode | Filter | Norm | Best Algorithm | Test Acc | Notes |
|---|-------|----------|------------|--------------|--------|------|----------------|----------|-------|
| 1 | both  | all      | YPR+Flex   | stats        | 5 Hz   | std  |                |          |       |

---
## Notes

**Suggested experiments to try:**

1. **Flex only** — Set `USE_YAW_PITCH_ROLL=False`, `USE_ACCELEROMETER=False`, `USE_FLEX_SENSORS=True`. See if flex alone is sufficient for static gestures.
2. **IMU only** — Opposite: disable flex, keep YPR + accel.
3. **Single hand** — Set `INCLUDE_HANDS=['left']` vs `['right']` to check hand asymmetry.
4. **FFT features** — Change `FEATURE_MODE='fft'` — useful for dynamic gestures (Flapping).
5. **No filter** — Set `APPLY_BUTTERWORTH=False` to see noise impact.
6. **Quaternion channels** — For `TwoHand_L_Flat_R_Flat`, enable `USE_QUATERNIONS=True` and disable YPR.
7. **Wrist only** — Reduce `INCLUDE_SEGMENTS=['wrist']` — minimal sensor configuration.